# 03. Florence-2 LoRA 파인튜닝

**목표**
- CORD v2로 Florence-2-base-ft LoRA 파인튜닝
- WandB로 학습 곡선 추적
- 체크포인트 Kaggle output에 저장

**실행 환경**: Kaggle (T4/P100)

In [ ]:
!rm -rf /kaggle/working/packages
!rm -rf /kaggle/working/korean-doc-understanding
!git clone https://github.com/shimtaehun/korean-doc-understanding.git

In [ ]:
import sys
sys.path.append("/kaggle/working/korean-doc-understanding")

In [ ]:
!pip install -q "transformers==4.47.0" "tokenizers==0.21.0" "jiwer"

In [ ]:
# 셀 4 — flash_attn mock (Kaggle에 flash_attn 없으므로 필수)
import sys
import types
import importlib.machinery
import transformers.utils.import_utils

for mod_name in ['flash_attn', 'flash_attn.flash_attn_interface', 'flash_attn.bert_padding']:
    mock = types.ModuleType(mod_name)
    mock.__spec__ = importlib.machinery.ModuleSpec(mod_name, loader=None)
    sys.modules[mod_name] = mock

transformers.utils.import_utils.is_flash_attn_2_available = lambda: False
print("flash_attn mock 완료")

In [ ]:
# 셀 5 — 기본 imports + GPU 확인
import os
import torch
import wandb
import yaml

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # fork 경고 억제

print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 셀 6 — WandB 로그인
from kaggle_secrets import UserSecretsClient

api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=api_key, relogin=True)

## 1. 학습 설정

In [ ]:
# 셀 7 — Config 로드 + Kaggle 환경 맞게 override
with open("/kaggle/working/korean-doc-understanding/configs/train_config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["output"]["checkpoint_dir"] = "/kaggle/working/checkpoints"
cfg["wandb"]["entity"] = "sthun0211-home"  # WandB 팀/사용자명
cfg["training"]["batch_size"] = 2
cfg["training"]["gradient_accumulation_steps"] = 8  # effective batch = 16
cfg["training"]["epochs"] = 15

print(yaml.dump(cfg, allow_unicode=True))

## 2. 모델 + 데이터 로드

In [ ]:
# 셀 8 — 모델 로드
from src.model.florence_lora import LoRASettings, load_florence_with_lora

lora_settings = LoRASettings(
    r=cfg["lora"]["r"],
    alpha=cfg["lora"]["alpha"],
    dropout=cfg["lora"]["dropout"],
    target_modules=cfg["lora"]["target_modules"],
)

model, processor = load_florence_with_lora(
    model_id=cfg["model"]["name"],
    lora_settings=lora_settings,
    device=DEVICE,
)

In [ ]:
# 셀 9 — 데이터셋 + DataLoader
from datasets import load_dataset
from torch.utils.data import DataLoader
from src.data.dataset import CORDDataset

hf_dataset = load_dataset(cfg["data"]["hf_dataset"])

train_ds = CORDDataset(hf_dataset["train"], processor, split="train",
                       max_length=cfg["model"]["max_length"], augment=True)
val_ds   = CORDDataset(hf_dataset["validation"], processor, split="validation",
                       max_length=cfg["model"]["max_length"], augment=False)

train_loader = DataLoader(train_ds, batch_size=cfg["training"]["batch_size"],
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg["training"]["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"train: {len(train_ds)}개 / val: {len(val_ds)}개")

## 3. 학습 실행

In [ ]:
# 셀 10 — Optimizer / Scheduler / Callback 설정
from torch.amp import GradScaler
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from src.training.evaluate import evaluate
from src.training.callbacks import WandBCallback, CheckpointCallback

wandb.init(
    project=cfg["wandb"]["project"],
    entity=cfg["wandb"]["entity"],
    config=cfg,
    name=f"lora-r{cfg['lora']['r']}-lr{cfg['training']['learning_rate']}-ep{cfg['training']['epochs']}-v2",
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=float(cfg["training"]["learning_rate"]),
    weight_decay=float(cfg["training"]["weight_decay"]),
)
total_steps  = len(train_loader) * cfg["training"]["epochs"]
warmup_steps = int(total_steps * cfg["training"]["warmup_ratio"])
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = GradScaler("cuda", enabled=cfg["training"]["fp16"])

wandb_cb = WandBCallback(log_interval=cfg["wandb"]["log_interval"])
ckpt_cb  = CheckpointCallback(
    checkpoint_dir=cfg["output"]["checkpoint_dir"],
    save_best_only=True,
    metric_name="cer",
    higher_is_better=False,  # CER: 낮을수록 좋음 → best 갱신마다 저장
)

print("설정 완료")

In [ ]:
# 셀 11 — 학습 루프
from torch.amp import autocast, GradScaler

accum_steps = cfg["training"]["gradient_accumulation_steps"]

for epoch in range(1, cfg["training"]["epochs"] + 1):
    print(f"\n=== Epoch {epoch}/{cfg['training']['epochs']} ===")
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        pixel_values   = batch["pixel_values"].to(DEVICE)
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        with autocast("cuda", dtype=torch.float16, enabled=cfg["training"]["fp16"]):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["training"]["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        wandb_cb.on_step_end(
            loss=total_loss / (step + 1),
            lr=scheduler.get_last_lr()[0],
            epoch=epoch,
        )

    val_metrics = evaluate(model, processor, val_loader, DEVICE)
    epoch_loss  = total_loss / len(train_loader)

    print(f"  loss={epoch_loss:.4f} | field_f1={val_metrics['field_f1']:.4f} | cer={val_metrics['cer']:.4f}")

    wandb_cb.on_epoch_end(epoch, epoch_loss, val_metrics)
    ckpt_cb.on_epoch_end(model, processor, epoch, val_metrics)

wandb_cb.on_train_end(ckpt_cb._best_metric)
print("\n학습 완료!")

## 4. 체크포인트 확인

In [ ]:
# 체크포인트 로드 (이전에 학습한 best_model 사용)
# ※ 이 셀부터 실행하면 재학습 없이 평가만 가능
# 셀 1~9(설치/설정/모델로드/데이터로드)는 실행해야 함, 학습 루프(셀 10~11)는 스킵

from peft import PeftModel

CKPT_PATH = "/kaggle/working/checkpoints/best_model"

model_eval, processor_eval = load_florence_with_lora(
    model_id=cfg["model"]["name"],
    lora_settings=lora_settings,
    device=DEVICE,
)
model_eval = PeftModel.from_pretrained(model_eval, CKPT_PATH)
model_eval.eval()
print(f"체크포인트 로드 완료: {CKPT_PATH}")

In [ ]:
# 수정된 evaluate로 F1 재측정
from src.training.evaluate import evaluate

val_metrics = evaluate(model_eval, processor_eval, val_loader, DEVICE)
print(f"Field F1: {val_metrics['field_f1']:.4f}")
print(f"CER:      {val_metrics['cer']:.4f}")

## 5. 체크포인트 로드 후 평가만 실행 (재학습 불필요)

In [ ]:
# 디버그 — 실제 예측 vs 정답 출력 (3개 샘플)
import torch
from torch.amp import autocast

debug_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

with torch.no_grad():
    for i, batch in enumerate(debug_loader):
        if i >= 3:
            break
        pixel_values   = batch["pixel_values"].to(DEVICE)
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        with autocast("cuda", dtype=torch.float16, enabled=cfg["training"]["fp16"]):
            output_ids = model_eval.generate(
                input_ids=input_ids,
                pixel_values=pixel_values,
                attention_mask=attention_mask,
                max_new_tokens=512,
                num_beams=3,
                early_stopping=False,
            )

        pred = processor_eval.batch_decode(output_ids, skip_special_tokens=True)[0]

        gt_ids = labels.clone()
        gt_ids[gt_ids == -100] = processor_eval.tokenizer.pad_token_id
        gt = processor_eval.batch_decode(gt_ids, skip_special_tokens=True)[0]

        print(f"=== 샘플 {i+1} ===")
        print(f"[예측] {pred[:300]}")
        print(f"[정답] {gt[:300]}")
        print(f"input_ids: {input_ids.shape}, output_ids: {output_ids.shape}")
        print()

In [ ]:
# 셀 12 — 저장된 체크포인트 목록 + 크기 확인
ckpt_dir = cfg["output"]["checkpoint_dir"]
for d in sorted(os.listdir(ckpt_dir)):
    size = sum(
        os.path.getsize(os.path.join(ckpt_dir, d, f))
        for f in os.listdir(os.path.join(ckpt_dir, d))
    ) / 1024 / 1024
    print(f"{d}: {size:.1f} MB")